In [2]:
!pip install gensim gradio
!python -m spacy download pt_core_news_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 43.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 59.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# --- ETAPA 1: Instalação das dependências ---
# !pip install gensim gradio
# !python -m spacy download pt_core_news_sm

import re
import numpy as np
import pandas as pd
import spacy
import gensim.downloader as api
from sklearn.tree import DecisionTreeClassifier
import gradio as gr

# 1. Carregamento do modelo spacy
print("Carregando modelo morfológico Spacy (pt_core_news_sm)...")
nlp = spacy.load("pt_core_news_sm")

# 2. Carregamento dos Word Embeddings (Gensim / GloVe de 50 dimensões)
print("Carregando espaço vetorial denso de embeddings (GloVe 50d)...")
word_vectors = api.load("glove-wiki-gigaword-50")

# 3. Base de Dados Supervisionada
dados_imobiliaria = [
    # Intenção: comprar_imovel
    ("Quero comprar um apartamento de 3 quartos com varanda", "comprar_imovel"),
    ("Gostaria de ver casas à venda no centro da cidade", "comprar_imovel"),
    ("Qual o preço médio para compra de cobertura com piscina?", "comprar_imovel"),
    ("Procuro imóvel residencial para comprar com financiamento", "comprar_imovel"),
    ("Vocês têm sobrados à venda na zona sul?", "comprar_imovel"),

    # Intenção: alugar_imovel
    ("Procurando kitnet para alugar perto da faculdade", "alugar_imovel"),
    ("Qual o valor do aluguel deste apartamento de 2 dormitórios?", "alugar_imovel"),
    ("Quero alugar um galpão comercial para minha empresa", "alugar_imovel"),
    ("Quais imóveis estão disponíveis para locação imediata?", "alugar_imovel"),
    ("Preciso de uma casa para alugar que aceite animais", "alugar_imovel"),

    # Intenção: suporte_manutencao
    ("O chuveiro do apartamento alugado queimou, como pedir conserto?", "suporte_manutencao"),
    ("Muro da casa está com infiltração e vazamento de água", "suporte_manutencao"),
    ("Preciso do contato do encanador para reparo na cozinha", "suporte_manutencao"),
    ("A porta da varanda quebrou, quem faz a manutenção?", "suporte_manutencao"),
    ("Vazamento no teto do banheiro precisa de reparo urgente", "suporte_manutencao"),

    # Intenção: 2via_boleto_contrato
    ("Como faço para baixar a segunda via do boleto do aluguel?", "2via_boleto_contrato"),
    ("Não recebi o boleto deste mês para pagamento", "2via_boleto_contrato"),
    ("Preciso do informe de rendimentos e cópia do contrato", "2via_boleto_contrato"),
    ("Onde pego o boleto atualizado com o valor do condomínio?", "2via_boleto_contrato"),
    ("Quero solicitar a segunda via do recibo de pagamento", "2via_boleto_contrato"),

    # Intenção: cancelar_contrato (LAB 04)
    ("Quero cancelar meu contrato de aluguel", "cancelar_contrato"),
    ("Como faço a rescisão do contrato de imóvel?", "cancelar_contrato"),
    ("Preciso encerrar o aluguel antes do prazo", "cancelar_contrato"),
    ("Quais as multas para devolução antecipada do imóvel?", "cancelar_contrato"),
    ("Solicito o distrato do contrato de compra", "cancelar_contrato")
]

df = pd.DataFrame(dados_imobiliaria, columns=["mensagem", "intencao"])
print(f"Dataset carregado com {len(df)} mensagens divididas em {df['intencao'].nunique()} intenções.")

def preprocessar_texto(texto: str) -> str:
    texto_limpo = texto.lower()
    texto_limpo = re.sub(r'[^a-záàâãéèêíïóôõöúçñ\s]', '', texto_limpo)
    doc = nlp(texto_limpo)
    tokens = [
        token.lemma_ for token in doc
        if not token.is_stop and not token.is_space and len(token.text) > 1
    ]
    return " ".join(tokens)

def extrair_sentence_embedding(texto_limpo: str, modelo_emb) -> np.ndarray:
    palavras = texto_limpo.split()
    vetores = [modelo_emb[p] for p in palavras if p in modelo_emb]

    if len(vetores) == 0:
        return np.zeros(modelo_emb.vector_size)

    # LAB 02: Max Pooling (np.max)
    return np.max(vetores, axis=0)

df['mensagem_limpa'] = df['mensagem'].apply(preprocessar_texto)

X_densos = np.array([
    extrair_sentence_embedding(txt, word_vectors) for txt in df['mensagem_limpa']
])

y = df['intencao'].values

# Treinamento com Decision Tree (LAB 01)
modelo_nlu = DecisionTreeClassifier(random_state=42)
modelo_nlu.fit(X_densos, y)
print("Modelo de Árvore de Decisão treinado com sucesso!")

# Base de Conhecimento
RESPOSTAS_PADRAO = {
    "comprar_imovel": (
        "**Atendimento de Vendas:** Ficamos felizes com seu interesse! "
        "Você pode conferir nosso catálogo de imóveis à venda em www.imobiliaria.com/vendas"
    ),
    "alugar_imovel": (
        "**Atendimento de Locação:** Temos ótimas opções disponíveis! "
        "Acesse www.imobiliaria.com/aluguel para filtrar por região."
    ),
    "suporte_manutencao": (
        "**Suporte e Manutenção:** Sentimos muito pelo inconveniente. "
        "Abra um chamado urgente em www.imobiliaria.com/manutencao"
    ),
    "2via_boleto_contrato": (
        "**Financeiro e Contratos:** Para acessar boletos, "
        "acesse a Área do Cliente em www.imobiliaria.com/cliente"
    ),
    "cancelar_contrato": (
        "**Distrato e Rescisão:** Para cancelamentos ou rescisão de contratos, "
        "acesse www.imobiliaria.com/distrato ou fale com nosso setor jurídico."
    )
}

# LAB 03: Limiar de Confiança alterado para 65% (0.65)
LIMIAR_CONFIANCA = 0.65

def processar_atendimento_sac(mensagem_usuario: str):
    if not mensagem_usuario or not mensagem_usuario.strip():
        return "N/A", "0.0%", "Aguardando mensagem...", "Aguardando entrada..."

    msg_limpa = preprocessar_texto(mensagem_usuario)
    vetor_input = extrair_sentence_embedding(msg_limpa, word_vectors).reshape(1, -1)

    probabilidades = modelo_nlu.predict_proba(vetor_input)[0]
    idx_maior_prob = np.argmax(probabilidades)
    confianca = probabilidades[idx_maior_prob]
    intencao_detectada = modelo_nlu.classes_[idx_maior_prob]

    percentual_confianca = f"{confianca * 100:.1f}%"

    # LAB 03: Validação com novo limiar
    if confianca >= LIMIAR_CONFIANCA:
        classificacao_status = f"IDENTIFICADO ({intencao_detectada})"
        texto_resposta = RESPOSTAS_PADRAO[intencao_detectada]
        intencao_exibida = intencao_detectada
    else:
        classificacao_status = "UNCERTAIN (Fallback Acionado - Corte 65%)"
        intencao_exibida = "Não Identificado"
        texto_resposta = (
            "Desculpe, não consegui compreender com clareza a sua solicitação. "
            "Estou transferindo para um de nossos atendentes."
        )

    card_resposta = f"""
    <div style="background-color: #f0f4f9; border-left: 5px solid #2b5c8f; padding: 15px; border-radius: 8px; margin-top: 10px;">
        <h4 style="margin: 0 0 8px 0; color: #2b5c8f;">Resposta Automática:</h4>
        <p style="margin: 0; font-size: 15px; color: #1a1a1a;">{texto_resposta}</p>
    </div>
    """

    return intencao_exibida, percentual_confianca, classificacao_status, card_resposta

# Interface Gradio
with gr.Blocks(theme=gr.themes.Soft(), title="SAC Imobiliário") as app:
    gr.Markdown("# SAC Imobiliário — ChatBot Inteligente (Labs 1 a 4)")

    with gr.Row():
        with gr.Column(scale=1):
            input_texto = gr.Textbox(lines=4, placeholder="Digite sua necessidade...", label="Mensagem")
            btn_processar = gr.Button("Processar Mensagem", variant="primary")

            gr.Examples(
                examples=[
                    ["Quero cancelar meu contrato de aluguel"],
                    ["Preciso de suporte técnico para vazamento."],
                    ["Quero ver apartamentos à venda."],
                    ["Vocês vendem terreno na Lua?"]
                ],
                inputs=input_texto
            )

        with gr.Column(scale=1):
            with gr.Row():
                out_intencao = gr.Textbox(label="Intenção", scale=2, interactive=False)
                out_confianca = gr.Textbox(label="Confiança", scale=1, interactive=False)

            out_status = gr.Textbox(label="Status da Decisão", interactive=False)
            out_resposta = gr.HTML(value="Aguardando mensagem...", label="Resposta")

    btn_processar.click(
        fn=processar_atendimento_sac,
        inputs=[input_texto],
        outputs=[out_intencao, out_confianca, out_status, out_resposta]
    )

app.launch(debug=True, share=True)

Carregando modelo morfológico Spacy (pt_core_news_sm)...
Carregando espaço vetorial denso de embeddings (GloVe 50d)...
[==================================================] 100.0% 66.0/66.0MB downloaded
Dataset carregado com 25 mensagens divididas em 5 intenções.
Modelo de Árvore de Decisão treinado com sucesso!


/tmp/ipykernel_548/2940340402.py:159: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="SAC Imobiliário") as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9a264fa0af26444647.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
